## Imports y configuración

In [30]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

spark = SparkSession.builder \
    .appName("P3_Validaciones") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER,
    "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE,
    "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE,
    "sfRole": SNOWFLAKE_ROLE,
    "use_arrow": "false"
}

print(f"Spark {spark.version} | Validando {SNOWFLAKE_DATABASE}.{SCHEMA_ANALYTICS}.OBT_TRIPS")

Spark 3.5.0 | Validando NYC_TLC.ANALYTICS.OBT_TRIPS


## Generar rangos de mes para procesamiento particionado

In [43]:
def generate_month_ranges(start_date, end_date):
    ranges = []
    current = start_date.replace(day=1)

    while current < end_date:
        if current.month == 12:
            next_month = current.replace(year=current.year + 1, month=1, day=1)
        else:
            next_month = current.replace(month=current.month + 1, day=1)

        ranges.append((current, next_month))
        current = next_month

    return ranges

start_date = datetime(2015, 1, 1)
end_date   = datetime(2025, 11, 30)

month_ranges = generate_month_ranges(start_date, end_date)

## Funcion de clasificación para cada check

In [35]:
def evaluate_check(name, invalid_count, total_rows, threshold_warning=0.01):
    if total_rows == 0:
        return {"check": name, "status": "WARNING", "details": "No rows to evaluate"}

    pct = invalid_count / total_rows

    if invalid_count == 0:
        return {"check": name, "status": "PASS", "details": "No issues found"}
    elif pct < threshold_warning:
        return {"check": name, "status": "WARNING", "details": f"{invalid_count} rows affected ({pct:.4%})"}
    else:
        return {"check": name, "status": "FAIL", "details": f"{invalid_count} rows affected ({pct:.4%})"}

## Métricas a ser probadas

In [41]:
def compute_metrics(df):
    return df.agg(
        F.count("*").alias("total_rows"),

        F.sum(F.when(F.col("PICKUP_DATETIME").isNull(), 1).otherwise(0)).alias("null_pickup"),
        F.sum(F.when(F.col("DROPOFF_DATETIME").isNull(), 1).otherwise(0)).alias("null_dropoff"),

        F.sum(F.when(F.col("DROPOFF_DATETIME") < F.col("PICKUP_DATETIME"), 1).otherwise(0)).alias("invalid_time"),

        F.sum(F.when(F.col("TRIP_DISTANCE") < 0, 1).otherwise(0)).alias("invalid_distance"),

        F.sum(F.when(
            (F.col("PASSENGER_COUNT") < 0) | (F.col("PASSENGER_COUNT") > 9),
            1
        ).otherwise(0)).alias("invalid_passengers"),

        F.sum(F.when(F.col("TOTAL_AMOUNT") < 0, 1).otherwise(0)).alias("invalid_total_amount")


    ).collect()[0]

## Función para construir reporte de calidad

In [39]:
def print_report(report):
    print("="*80)
    print(f"VALIDATION REPORT - {report['year_month']}")
    print("="*80)

    total = report["total_rows"]
    print(f"Total Rows: {total}")
    print("-"*80)

    pass_count = sum(1 for c in report["checks"] if c["status"] == "PASS")
    warn_count = sum(1 for c in report["checks"] if c["status"] == "WARNING")
    fail_count = sum(1 for c in report["checks"] if c["status"] == "FAIL")

    print(f"PASS: {pass_count} | WARNING: {warn_count} | FAIL: {fail_count}")
    print("-"*80)

    for c in report["checks"]:
        icon = "✅" if c["status"] == "PASS" else "⚠️" if c["status"] == "WARNING" else "❌"

        print(f"{icon} {c['check']}")
        print(f"   Status: {c['status']}")
        print(f"   Details: {c['details']}")
        print()

    print("="*80)

In [42]:
def build_report(metrics, year_month):
    total = metrics["total_rows"]

    checks = [
        evaluate_check("Null check: PICKUP_DATETIME", metrics["null_pickup"], total),
        evaluate_check("Null check: DROPOFF_DATETIME", metrics["null_dropoff"], total),
        evaluate_check("Time consistency: dropoff > pickup", metrics["invalid_time"], total),
        evaluate_check("Range: TRIP_DISTANCE >= 0", metrics["invalid_distance"], total),
        evaluate_check("Range: PASSENGER_COUNT 0-9", metrics["invalid_passengers"], total),
        evaluate_check("Range: TOTAL_AMOUNT >= 0", metrics["invalid_total_amount"], total),

    ]

    return {
        "year_month": year_month,
        "total_rows": total,
        "checks": checks
    }

In [44]:
def print_report(report):
    print("="*80)
    print(f"VALIDATION REPORT - {report['year_month']}")
    print("="*80)

    total = report["total_rows"]
    print(f"Total Rows: {total}")
    print("-"*80)

    pass_count = sum(1 for c in report["checks"] if c["status"] == "PASS")
    warn_count = sum(1 for c in report["checks"] if c["status"] == "WARNING")
    fail_count = sum(1 for c in report["checks"] if c["status"] == "FAIL")

    print(f"PASS: {pass_count} | WARNING: {warn_count} | FAIL: {fail_count}")
    print("-"*80)

    for c in report["checks"]:
        icon = "✅" if c["status"] == "PASS" else "⚠️" if c["status"] == "WARNING" else "❌"

        print(f"{icon} {c['check']}")
        print(f"   Status: {c['status']}")
        print(f"   Details: {c['details']}")
        print()

    print("="*80)

In [46]:
###### all_reports = []

for start, end in month_ranges:
    ym = start.strftime("%Y-%m")

    try:
        print(f"Processing {ym}")

        query = f"""
        SELECT *
        FROM {SNOWFLAKE_DATABASE}.{SCHEMA_ANALYTICS}.OBT_TRIPS
        WHERE PICKUP_DATETIME >= '{start.strftime("%Y-%m-%d %H:%M:%S")}'
          AND PICKUP_DATETIME < '{end.strftime("%Y-%m-%d %H:%M:%S")}'
        """

        df = spark.read \
            .format("net.snowflake.spark.snowflake") \
            .options(**sf_options) \
            .option("query", query) \
            .load()

        metrics = compute_metrics(df)
        report = build_report(metrics, ym)

        print_report(report)

        del df
        del metrics
        del report

    except Exception as e:
        print(f"Error processing {ym}: {e}")

print("Procesamiento terminado")


Processing 2015-01
VALIDATION REPORT - 2015-01
Total Rows: 14242947
--------------------------------------------------------------------------------
PASS: 6 | WARNING: 0 | FAIL: 0
--------------------------------------------------------------------------------
✅ Null check: PICKUP_DATETIME
   Status: PASS
   Details: No issues found

✅ Null check: DROPOFF_DATETIME
   Status: PASS
   Details: No issues found

✅ Time consistency: dropoff > pickup
   Status: PASS
   Details: No issues found

✅ Range: TRIP_DISTANCE >= 0
   Status: PASS
   Details: No issues found

✅ Range: PASSENGER_COUNT 0-9
   Status: PASS
   Details: No issues found

✅ Range: TOTAL_AMOUNT >= 0
   Status: PASS
   Details: No issues found

Processing 2015-02
VALIDATION REPORT - 2015-02
Total Rows: 14010599
--------------------------------------------------------------------------------
PASS: 6 | WARNING: 0 | FAIL: 0
--------------------------------------------------------------------------------
✅ Null check: PICKUP_DATET